# Module 11 — DETR Family

DETR reformulates object detection as a set prediction problem,
eliminating anchors, NMS, and hand-crafted components.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matcher import HungarianMatcher, box_cxcywh_to_xyxy, generalized_box_iou
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Hungarian Matching

The Hungarian algorithm finds the minimum-cost one-to-one assignment.

In [ ]:
from scipy.optimize import linear_sum_assignment
import numpy as np

# Simple example: 3 predictions, 2 GT objects
cost = np.array([[0.8, 3.0],  # pred 0: close to GT 0
                  [2.5, 0.3],  # pred 1: close to GT 1
                  [4.0, 5.0]]) # pred 2: far from both
row_idx, col_idx = linear_sum_assignment(cost)
print('Prediction indices:', row_idx)
print('GT indices:        ', col_idx)
print('Total cost:        ', cost[row_idx, col_idx].sum())

## 2. Mini DETR Architecture

In [ ]:
class MiniDETR(nn.Module):
    def __init__(self, num_classes=80, num_queries=100, d_model=256, nhead=8, num_encoder=6, num_decoder=6):
        super().__init__()
        from torchvision.models import resnet50
        backbone = resnet50(weights=None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])  # up to layer4
        self.input_proj = nn.Conv2d(2048, d_model, 1)
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead,
                                           num_encoder_layers=num_encoder,
                                           num_decoder_layers=num_decoder)
        self.query_embed = nn.Embedding(num_queries, d_model)
        self.class_head = nn.Linear(d_model, num_classes + 1)  # +1 for no-object
        self.bbox_head  = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(),
                                         nn.Linear(d_model, 4), nn.Sigmoid())

    def forward(self, x):
        B = x.shape[0]
        feat = self.input_proj(self.backbone(x))  # (B, d_model, H, W)
        H, W = feat.shape[-2:]
        d_model = feat.shape[1]
        # Flatten spatial dims
        src = feat.flatten(2).permute(2, 0, 1)   # (HW, B, d_model)
        tgt = self.query_embed.weight.unsqueeze(1).repeat(1, B, 1)  # (Q, B, d_model)
        out = self.transformer(src, tgt)           # (Q, B, d_model)
        out = out.permute(1, 0, 2)                 # (B, Q, d_model)
        return self.class_head(out), self.bbox_head(out)

detr = MiniDETR(num_classes=80, num_queries=20)
x = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    cls_out, box_out = detr(x)
print('Class logits:', cls_out.shape)  # (1, 20, 81)
print('Box preds:', box_out.shape)     # (1, 20, 4)

## 3. Generalised IoU

GIoU penalises predictions that don't overlap with GT but are far away.

In [ ]:
# 3 predictions, 2 GT boxes (normalised cxcywh)
pred_boxes_cxcywh = torch.tensor([
    [0.5, 0.5, 0.3, 0.4],  # overlaps GT 0
    [0.2, 0.3, 0.1, 0.2],  # overlaps GT 1
    [0.9, 0.9, 0.1, 0.1],  # no overlap
])
gt_boxes_cxcywh = torch.tensor([
    [0.5, 0.5, 0.4, 0.4],
    [0.2, 0.3, 0.15, 0.25],
])
pred_xyxy = box_cxcywh_to_xyxy(pred_boxes_cxcywh)
gt_xyxy   = box_cxcywh_to_xyxy(gt_boxes_cxcywh)
giou = generalized_box_iou(pred_xyxy, gt_xyxy)
print('GIoU matrix:')
print(giou.round(decimals=3))

## Exercise — DETR Bipartite Loss

Implement `detr_loss(pred_logits, pred_boxes, tgt_classes, tgt_boxes, matcher)`
that:
1. Uses `HungarianMatcher` to find the assignment.
2. Computes cross-entropy loss for matched classes.
3. Computes L1 + GIoU loss for matched boxes.

In [ ]:
### EXERCISE
def detr_loss(pred_logits, pred_boxes, tgt_classes, tgt_boxes, matcher,
             lambda_bbox=5.0, lambda_giou=2.0):
    """
    Compute DETR set prediction loss.
    Returns: dict with keys 'loss_ce', 'loss_bbox', 'loss_giou', 'total'.
    """
    # TODO
    raise NotImplementedError